# CIC-IDS-2017 model comparison

This notebook reads completed MLflow runs; it does not train models. Change the round filter, restart the kernel, and run all cells so every table is produced from the same query.

## 1. Select experiment rounds

In [ ]:
ROUND_FILTER = ["baseline", "tree"]

# Examples:
# ROUND_FILTER = ["tree"]
# ROUND_FILTER = ["baseline", "neural"]
# ROUND_FILTER = ["baseline", "tree", "neural"]
# ROUND_FILTER = None  # all rounds

DATASET_VERSION = None  # optionally select one sha256:... dataset version

## 2. Imports and filter validation

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ids_ml.reporting import (
    best_per_family,
    build_screening_report,
    download_csv_artifact,
    feature_set_comparison,
    screening_candidates,
    weighting_comparison,
)
from ids_ml.specs import validate_round_filter

SELECTED_ROUNDS = validate_round_filter(ROUND_FILTER)
print(f"Selected rounds: {', '.join(SELECTED_ROUNDS)}")

## 3. Load matching MLflow runs and check coverage

In [ ]:
report = build_screening_report(SELECTED_ROUNDS, dataset_version=DATASET_VERSION)

if report.contract:
    display(pd.Series(report.contract, name="Selected experiment contract").to_frame())
else:
    print("No completed run is available to establish a dataset and split contract.")

if len(report.available_contracts) > 1:
    print("Other compatible contracts are available (ranked by coverage, then recency):")
    display(report.available_contracts)

display(report.coverage)

if not report.duplicate_runs.empty:
    print("Configurations with more than one matching successful run:")
    display(report.duplicate_runs)
else:
    print("No duplicate successful configurations were found.")

if not report.failed_runs.empty:
    print("Failed or interrupted matching runs:")
    display(report.failed_runs[[
        "screening_round", "run_id", "run_name", "configuration_key", "status", "start_time"
    ]])
else:
    print("No failed or interrupted matching runs were found.")

The contract prevents results from different dataset versions, fit/validation partitions, or timing samples from being compared as if they came from the same experiment. Coverage identifies rounds that are selected but not yet complete.

## 4. Complete validation leaderboard

In [ ]:
leaderboard = report.leaderboard.copy()
if leaderboard.empty:
    print("No matching successful configurations are available.")
else:
    display(leaderboard)

Every matching latest configuration is displayed. Ranking uses validation macro F1 only; the remaining metrics describe accuracy, attack-detection behavior, cost, and latency tradeoffs.

## 5. Feature-set, weighting, and family comparisons

In [ ]:
feature_comparison = feature_set_comparison(leaderboard)
weight_comparison = weighting_comparison(leaderboard)
family_best = best_per_family(leaderboard)

print("71 versus 64 source features:")
display(feature_comparison if not feature_comparison.empty else pd.DataFrame())

print("Balanced versus unweighted training:")
display(weight_comparison if not weight_comparison.empty else pd.DataFrame())

print("Best validation configuration per family and experiment round:")
display(family_best if not family_best.empty else pd.DataFrame())

## 6. Macro F1 versus operational performance

In [ ]:
if leaderboard.empty:
    print("No runs are available for operational plots.")
else:
    figure, axes = plt.subplots(1, 2, figsize=(15, 6))
    for family, rows in leaderboard.groupby("model_family"):
        axes[0].scatter(rows["latency_p99_ms"], rows["macro_f1"], label=family)
        axes[1].scatter(rows["throughput_flows_per_second"], rows["macro_f1"], label=family)
    axes[0].set_xlabel("Complete-pipeline p99 latency (ms, CPU)")
    axes[0].set_ylabel("Validation macro F1")
    axes[0].set_title("Validation performance versus latency")
    axes[1].set_xlabel("Complete-pipeline throughput (flows/second, CPU)")
    axes[1].set_ylabel("Validation macro F1")
    axes[1].set_title("Validation performance versus throughput")
    axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    figure.tight_layout()
    plt.show()

## 7. Screening shortlist

In [ ]:
candidates = screening_candidates(leaderboard)
if candidates.empty:
    print(
        "The four-category shortlist requires completed boosting, bagging, and neural candidates "
        "among the selected rounds."
    )
else:
    candidate_columns = [
        "candidate_category", "screening_round", "model_family", "weighting_mode",
        "feature_set", "run_id", "macro_f1", "binary_attack_recall",
        "latency_p99_ms", "throughput_flows_per_second",
    ]
    display(candidates[candidate_columns])

## 8. Candidate per-class reports and confusion matrices

In [ ]:
if candidates.empty:
    print("No complete shortlist is available, so candidate artifacts were not downloaded.")
else:
    candidate_reports = []
    for candidate in candidates.drop_duplicates("run_id").itertuples(index=False):
        per_class = download_csv_artifact(
            candidate.run_id, "evaluation/per_class_report.csv"
        )
        per_class.insert(0, "run_id", candidate.run_id)
        per_class.insert(0, "model_family", candidate.model_family)
        candidate_reports.append(per_class)
    display(pd.concat(candidate_reports, ignore_index=True))

    for candidate in candidates.drop_duplicates("run_id").itertuples(index=False):
        print(f"{candidate.model_family} — {candidate.run_id}")
        raw_matrix = download_csv_artifact(
            candidate.run_id, "evaluation/confusion_matrix_raw.csv"
        )
        normalized_matrix = download_csv_artifact(
            candidate.run_id, "evaluation/confusion_matrix_row_normalized.csv"
        )
        print("Raw confusion matrix")
        display(raw_matrix)
        print("Row-normalized confusion matrix")
        display(normalized_matrix)

## 9. Tree diagnostics for family leaders

In [ ]:
tree_leaders = family_best.loc[family_best["screening_round"].eq("tree")]
if tree_leaders.empty:
    print("No selected tree challenger results are available.")
else:
    for leader in tree_leaders.itertuples(index=False):
        try:
            importance = download_csv_artifact(
                leader.run_id, "diagnostics/tree_feature_importance.csv"
            )
        except Exception as error:
            print(f"Could not load importance for {leader.model_family}: {error}")
            continue
        print(f"{leader.model_family}: top 20 model-reported importances")
        display(importance.head(20))

## 10. Neural diagnostics for family leaders

In [ ]:
neural_leaders = family_best.loc[family_best["screening_round"].eq("neural")]
if neural_leaders.empty:
    print("No selected neural challenger results are available.")
else:
    for leader in neural_leaders.itertuples(index=False):
        try:
            history = download_csv_artifact(
                leader.run_id, "training/epoch_selection_history.csv"
            )
        except Exception as error:
            print(f"Could not load training history for {leader.model_family}: {error}")
            continue
        print(
            f"{leader.model_family}: selected epochs = "
            f"{leader.selected_epochs if pd.notna(leader.selected_epochs) else 'unavailable'}"
        )
        display(history)
        metric_columns = [
            column for column in ["training_loss", "stopping_macro_f1"] if column in history
        ]
        if metric_columns:
            history[metric_columns].plot(
                figsize=(9, 4), title=f"{leader.model_family} epoch-selection history"
            )
            plt.show()

No result in this notebook evaluates the protected test partition or selects a final deployable model. Final tuning and the one-time test evaluation remain separate later stages.